# Win Rate Repair and Current Win Rate Check
This notebook repairs missing base prices in the database and computes the latest win rate.

In [3]:
import os
from pathlib import Path
import json
import subprocess

workspace = Path(r'c:\Project rohan\Alpha_Lens')
os.chdir(workspace)
python_exe = workspace / '.alpha-venv' / 'Scripts' / 'python.exe'
script = workspace / 'backend' / 'win_rate_check.py'
proc = subprocess.run([
    str(python_exe),
    str(script)
], capture_output=True, text=True, encoding='utf-8')
print(proc.stdout)
if proc.stderr:
    print('STDERR:')
    print(proc.stderr)
report_path = workspace / 'backend' / 'win_rate_report.json'
if report_path.exists():
    with open(report_path, 'r', encoding='utf-8') as f:
        report = json.load(f)
    print('---')
    print('Win rate report summary:')
    print(json.dumps({
        'generated_at': report.get('generated_at'),
        'total_signals': report.get('total_signals'),
        'resolved': report.get('resolved'),
        'win_rate_pct': report.get('win_rate_pct'),
        'wins': report.get('wins'),
        'losses': report.get('losses'),
        'expired': report.get('expired'),
        'no_data': report.get('no_data'),
    }, indent=2))
else:
    print('No report file generated.')



   ALPHA LENS — WIN RATE CHECKER (Last 100 Signals)
   Target: +2.0%  |  Stop: -1.0%  |  Window: 3 days

 repaired base_price for 0 signals.
Found 100 signals to evaluate...

[1/100] HCLTECH.NS BULLISH @ Rs.1200.00  |  Opening Bell: Sensex starts 100 pts higher, Nifty ...
   [LOSS]  -0.99%  Day 2

[2/100] COALINDIA.NS BULLISH @ Rs.480.00  |  Coal India Shares In Focus As Motilal Oswal Mainta...
   [LOSS]  -1.07%  Day 1

[3/100] MCX.NS BULLISH @ Rs.2912.90  |  Is Stock Market Closed Today? NSE, BSE Shut; MCX O...
   [LOSS]  -1.03%  Day 1

[4/100] BSE.NS BULLISH @ Rs.3709.40  |  Is Stock Market Closed Today? NSE, BSE Shut; MCX O...
   [WIN]  +2.11%  Day 1

[5/100] AMBUJACEM.NS BULLISH @ Rs.444.60  |  Ambuja Cements Q4 Results: Cons profit surges 78% ...
   [LOSS]  -1.68%  Day 1

[6/100] SBIN.NS BEARISH @ Rs.1018.50  |  Weak margins, treasury income drag SBI shares lowe...
   [WIN]  -2.59%  Day 2

[7/100] SBIN.NS BULLISH @ Rs.1018.50  |  SBI Q4 profit rises 6%, investors cautious over bo

In [4]:
import yfinance as yf
from datetime import datetime

print('Running RELIANCE.NS check')
ticker = 'RELIANCE.NS'
start = '2026-05-14'
end = '2026-05-17'
base = 1335.90

t = yf.Ticker(ticker)
hist = t.history(start=start, end=end, interval='15m')
print('rows', len(hist))
print(hist[['High','Low','Close']].tail())
print('max high', hist['High'].max())
print('min low', hist['Low'].min())
print('last close', hist['Close'].iloc[-1])
for idx, row in hist.iterrows():
    high_pct = (row['High'] - base) / base * 100
    low_pct = (row['Low'] - base) / base * 100
    if high_pct >= 1.0:
        print('HIGH HIT', idx, high_pct)
    if low_pct <= -2.0:
        print('LOW HIT target', idx, low_pct)


Running RELIANCE.NS check
rows 50
                                  High          Low        Close
Datetime                                                        
2026-05-15 14:15:00+05:30  1336.500000  1334.699951  1335.800049
2026-05-15 14:30:00+05:30  1336.699951  1335.000000  1335.599976
2026-05-15 14:45:00+05:30  1335.599976  1329.400024  1331.199951
2026-05-15 15:00:00+05:30  1337.900024  1330.500000  1337.300049
2026-05-15 15:15:00+05:30  1340.000000  1336.000000  1339.000000
max high 1377.800048828125
min low 1329.4000244140625
last close 1339.0
HIGH HIT 2026-05-14 09:15:00+05:30 2.5151583202335437
HIGH HIT 2026-05-14 09:30:00+05:30 2.447786180547751
HIGH HIT 2026-05-14 09:45:00+05:30 2.140878808294027
HIGH HIT 2026-05-14 10:00:00+05:30 1.9836832408161094
HIGH HIT 2026-05-14 10:15:00+05:30 2.036078717414283
HIGH HIT 2026-05-14 10:30:00+05:30 2.253162661875882
HIGH HIT 2026-05-14 10:45:00+05:30 2.021111192010061
HIGH HIT 2026-05-14 11:00:00+05:30 1.9986507662203314
HIGH HIT 202

In [5]:
import sqlite3

db_path = r'c:\Project rohan\Alpha_Lens\news_cache.db'
conn = sqlite3.connect(db_path)
c = conn.cursor()

# Total signals
total = c.execute('SELECT COUNT(*) FROM stock_impact').fetchone()[0]

# By status
statuses = c.execute('SELECT status, COUNT(*) FROM stock_impact GROUP BY status').fetchall()

conn.close()

print(f'Total Signals: {total}\n')
for status, count in sorted(statuses):
    print(f'{status}: {count}')


Total Signals: 342

Active View: 342


In [6]:
import subprocess
from pathlib import Path

workspace = Path(r'c:\Project rohan\Alpha_Lens')
python_exe = workspace / '.alpha-venv' / 'Scripts' / 'python.exe'
script = workspace / 'backend' / 'remediate_signals.py'

proc = subprocess.run([str(python_exe), str(script)], capture_output=True, text=True, timeout=300)
print(proc.stdout)
if proc.stderr:
    print('STDERR:', proc.stderr)



STDERR: Traceback (most recent call last):
  File "c:\Project rohan\Alpha_Lens\backend\remediate_signals.py", line 11, in <module>
    from backend import yfinance_twelvedata_shim as yf
ModuleNotFoundError: No module named 'backend'



In [ ]:
import subprocess
from pathlib import Path

workspace = Path(r'c:\Project rohan\Alpha_Lens')
python_exe = workspace / '.alpha-venv' / 'Scripts' / 'python.exe'
script = workspace / 'backend' / 'remediate_signals.py'

proc = subprocess.run([str(python_exe), str(script)], capture_output=True, text=True, timeout=300, cwd=str(workspace))
print(proc.stdout)
if proc.stderr:
    print('STDERR:', proc.stderr)
print('Return code:', proc.returncode)
